# 01B. Outlier Detection & Descriptive Statistics
**Goal:** Calculate descriptive statistics (mean, median, mode, RMS, Std) and perform outlier detection on sensor data.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

os.makedirs('plots', exist_ok=True)

# Column names
cols = ['unit', 'cycle', 'op1', 'op2', 'op3'] + [f's{i}' for i in range(1, 22)]

# Load Data
train_df = pd.read_csv('../data/train_FD001.txt', sep='\\s+', header=None, names=cols)
sensors = [f's{i}' for i in range(1, 22)]
print(f"Shape: {train_df.shape}")
if not train_df.empty:
    display(train_df.head())

## 1. Descriptive Statistics
Calculating **Mean**, **Median**, **Mode**, **RMS** (Root Mean Square), and **Standard Deviation** for all 21 sensors.

In [ ]:
stats_df = pd.DataFrame(index=sensors)

# 1. Mean
stats_df['Mean'] = train_df[sensors].mean()

# 2. Median
stats_df['Median'] = train_df[sensors].median()

# 3. Mode (taking the first mode if multiple exist)
stats_df['Mode'] = train_df[sensors].mode().iloc[0]

# 4. Standard Deviation
stats_df['Std_Dev'] = train_df[sensors].std()

# 5. RMS (Root Mean Square) = sqrt(mean(x^2))
stats_df['RMS'] = np.sqrt((train_df[sensors]**2).mean())

display(stats_df)

## 2. Visual Outlier Detection (Boxplots)
Using boxplots to visualize the spread and outliers for each sensor.

In [ ]:
plt.figure(figsize=(20, 8))
sns.boxplot(data=train_df[sensors])
plt.title('Sensor Value Distributions (Detecting Outliers)')
plt.xticks(rotation=45)
plt.savefig('plots/sensor_boxplots.png')
plt.show()

## 3. Z-Score Outlier Detection
Detecting points that are more than 3 standard deviations away from the mean.

In [ ]:
z_scores = np.abs((train_df[sensors] - train_df[sensors].mean()) / train_df[sensors].std())
outliers_z = (z_scores > 3)

print("Number of outliers per sensor using Z-Score > 3:")
display(outliers_z.sum().to_frame(name="Outlier Count").T)

## 4. IQR (Interquartile Range) Method
Calculating boundaries Q1 - 1.5*IQR and Q3 + 1.5*IQR.

In [ ]:
Q1 = train_df[sensors].quantile(0.25)
Q3 = train_df[sensors].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers_iqr = ((train_df[sensors] < lower_bound) | (train_df[sensors] > upper_bound))

print("Number of outliers per sensor using IQR method:")
display(outliers_iqr.sum().to_frame(name="Outlier Count").T)

## 5. Handling Outliers (Clipping)
Instead of dropping rows (which destroys the time-series sequence), we cap the extreme values to the IQR or Z-score boundaries so that noise is reduced.

In [ ]:
df_clipped = train_df.copy()

for col in sensors:
    df_clipped[col] = np.clip(df_clipped[col], lower_bound[col], upper_bound[col])

print("Outliers have been clipped to the IQR boundaries. Validation of maximum values after clipping:")
display(df_clipped[sensors].max().to_frame(name='Max After Clipping').T)